In [0]:
%sql
-- Silver involves several decisions: what type each column gets, how to handle hidden values, which columns become dimension tables. Rather than guess, we’ll query bronze first and let the results drive those decisions.

SELECT 'GE65_Sprsn_Flag' AS flag_column, GE65_Sprsn_Flag AS value, COUNT(*) AS n
FROM cms.bronze.partd_prescriber_drug
GROUP BY GE65_Sprsn_Flag

UNION ALL

SELECT 'GE65_Bene_Sprsn_Flag', GE65_Bene_Sprsn_Flag, COUNT(*)
FROM cms.bronze.partd_prescriber_drug
GROUP BY GE65_Bene_Sprsn_Flag

ORDER BY flag_column, n DESC;

-- For this, we're asking what values the flags actually take. Most are just * or #. Some are missing. 

In [0]:
%sql
SELECT
  SUM(CASE WHEN Tot_Clms          IS NOT NULL AND try_cast(Tot_Clms          AS BIGINT)        IS NULL THEN 1 ELSE 0 END) AS bad_tot_clms,
  SUM(CASE WHEN Tot_30day_Fills   IS NOT NULL AND try_cast(Tot_30day_Fills   AS DECIMAL(18,1)) IS NULL THEN 1 ELSE 0 END) AS bad_tot_30day,
  SUM(CASE WHEN Tot_Day_Suply     IS NOT NULL AND try_cast(Tot_Day_Suply     AS BIGINT)        IS NULL THEN 1 ELSE 0 END) AS bad_day_suply,
  SUM(CASE WHEN Tot_Drug_Cst      IS NOT NULL AND try_cast(Tot_Drug_Cst      AS DECIMAL(18,2)) IS NULL THEN 1 ELSE 0 END) AS bad_drug_cst,
  SUM(CASE WHEN Tot_Benes         IS NOT NULL AND try_cast(Tot_Benes         AS BIGINT)        IS NULL THEN 1 ELSE 0 END) AS bad_tot_benes,
  SUM(CASE WHEN GE65_Tot_Drug_Cst IS NOT NULL AND try_cast(GE65_Tot_Drug_Cst AS DECIMAL(18,2)) IS NULL THEN 1 ELSE 0 END) AS bad_ge65_cst
FROM cms.bronze.partd_prescriber_drug;

-- Now we ask will every numeric column convert cleanly? 

In [0]:
%sql
SELECT
  COUNT(*)                                            AS total_rows,
  SUM(CASE WHEN Tot_Benes IS NULL THEN 1 ELSE 0 END)  AS tot_benes_hidden,
  ROUND(100 * AVG(CASE WHEN Tot_Benes IS NULL THEN 1 ELSE 0 END), 1) AS pct_benes_hidden,
  SUM(CASE WHEN Tot_Drug_Cst IS NULL THEN 1 ELSE 0 END) AS drug_cst_missing
FROM cms.bronze.partd_prescriber_drug;

-- How much is hidden overall.

In [0]:
%sql
SELECT
  COUNT(DISTINCT Prscrbr_NPI)   AS distinct_prescribers,
  COUNT(DISTINCT Brnd_Name)     AS distinct_brands,
  COUNT(DISTINCT Gnrc_Name)     AS distinct_generics,
  COUNT(DISTINCT Prscrbr_Type)  AS distinct_specialties,
  COUNT(DISTINCT Prscrbr_State_Abrvtn) AS distinct_states,
  MIN(LENGTH(Prscrbr_NPI))      AS min_npi_len,
  MAX(LENGTH(Prscrbr_NPI))      AS max_npi_len
FROM cms.bronze.partd_prescriber_drug;

-- How many distinct values does each dimension have?

In [0]:
%sql
SELECT Prscrbr_Type, COUNT(*) AS n_rows, COUNT(DISTINCT Prscrbr_NPI) AS n_prescribers
FROM cms.bronze.partd_prescriber_drug
WHERE lower(Prscrbr_Type) LIKE '%dent%'
   OR lower(Prscrbr_Type) LIKE '%oral%'
GROUP BY Prscrbr_Type
ORDER BY n_rows DESC;

-- Dental prescrbers